# LLM auto-grading -- Kaggle runner

Clones the public code repo ([https://github.com/ptran1203/msc.git](https://github.com/ptran1203/msc.git)) and attaches the private data
via the Kaggle dataset `ptran1203/llm-autograding-course-data` (must be added under Notebook ->
Add Input before running). Course rules forbid distributing the real student data
outside the class, so it lives only in that private dataset, never in the repo itself.

Edit the **config cell** below, then Run All.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
%cd /kaggle/working
!rm -rf msc
!git clone --depth 1 https://github.com/ptran1203/msc.git
%cd /kaggle/working/msc/llm
!pip install -q -r requirements.txt


In [ ]:
import shutil, pathlib
dataset_root = pathlib.Path("/kaggle/input/llm-autograding-course-data")
repo_llm = pathlib.Path("/kaggle/working/msc/llm")
shutil.copytree(dataset_root / "data", repo_llm / "data")
(repo_llm / "synth").mkdir(exist_ok=True)
shutil.copytree(dataset_root / "synth_generated", repo_llm / "synth" / "generated")
print("data staged OK")


## Config -- edit this, then run the cell below

- `SPLIT`: `"sample"` (32 labeled, validates against gold) or `"public"` (328 unlabeled, the real submission target)
- `TASKS`: comma-separated task numbers, e.g. `"1,2"`
- `MODE`: `"zeroshot"`, `"fewshot"` (static hand-picked exemplars), or `"rag"` (retrieval from the 89-sample pool)

In [ ]:
SPLIT = "sample"
TASKS = "1,2"
MODE = "rag"


In [ ]:
%cd /kaggle/working/msc/llm/src
mode_flag = {"zeroshot": "", "fewshot": "--fewshot", "rag": "--rag"}[MODE]
!python -u run_prompting.py --split {SPLIT} --tasks {TASKS} {mode_flag}


## Output

Predictions and logs are written to `/kaggle/working/msc/llm/outputs/` -- anything under
`/kaggle/working` is saved as this notebook's output automatically on commit.